# Pipeline NLP en español con NLTK

Procesamiento de la columna `review_summary_es`.

**Orden del pipeline**
1. Tokenización
2. Etiquetado POS
3. Borrado de stopwords
4. NER
5. Mayúsculas / minúsculas
6. Lematización

## 2. Clase `NLTK_POS_Reviews`

Etiquetas POS (CESS-ESP / EAGLES) de referencia:

| Etiqueta | Significado |
|---|---|
| `ncms000` | Sustantivo común masculino singular |
| `ncfp000` | Sustantivo común femenino plural |
| `np00000` | Nombre propio |
| `vmip3s0` | Verbo principal, indicativo presente, 3ª pers. singular |
| `aq0cs0` | Adjetivo calificativo |
| `rg` | Adverbio |
| `sp` | Preposición |
| `da0ms0` | Artículo determinado masculino singular |

In [ ]:
class NLTK_POS_Reviews:
    """Pipeline de preprocesamiento lingüístico para texto en español."""

    # Mapeo de tags EAGLES (cess_esp) a etiquetas legibles
    _MAPA_POS = {
        "n": "NOUN", "a": "ADJ", "v": "VERB", "r": "ADV", "d": "DET",
        "p": "PRON", "s": "ADP", "c": "CONJ", "i": "INTJ", "z": "NUM",
        "w": "DATE", "f": "PUNCT",
    }

    def __init__(self, ruta_csv, columna="review_summary_es",
                 dir_cache="cache_nltk", max_sents_ner=None):
        self.ruta_csv = ruta_csv
        self.columna = columna
        self.dir_cache = dir_cache
        self.max_sents_ner = max_sents_ner
        self.df = None

        os.makedirs(self.dir_cache, exist_ok=True)
        self._descargar_recursos()

        self.stop_es = set(stopwords.words("spanish"))
        self.stemmer = SnowballStemmer("spanish")  # respaldo si no hay simplemma
        self.tagger_pos = self._cargar_o_entrenar("pos.pkl", self._entrenar_pos)
        self.tagger_ner = self._cargar_o_entrenar("ner.pkl", self._entrenar_ner)

    # ------------------------------------------------------------------ setup
    @staticmethod
    def _descargar_recursos():
        for recurso in ["punkt", "punkt_tab", "stopwords", "cess_esp", "conll2002"]:
            nltk.download(recurso, quiet=True)

    def _cargar_o_entrenar(self, nombre, funcion_entrenamiento):
        ruta = os.path.join(self.dir_cache, nombre)
        if os.path.exists(ruta):
            with open(ruta, "rb") as f:
                return pickle.load(f)
        modelo = funcion_entrenamiento()
        with open(ruta, "wb") as f:
            pickle.dump(modelo, f)
        return modelo

    def _simplificar_tag(self, tag):
        """'np00000' -> 'PROPN', 'nc0s000' -> 'NOUN', 'vmip3s0' -> 'VERB', etc."""
        t = tag.lower()
        if t.startswith("np"):
            return "PROPN"
        if t.startswith("va") or t.startswith("vs"):
            return "AUX"
        return self._MAPA_POS.get(t[:1], "X")

    # ---------------------------------------------------------- entrenamientos
    def _entrenar_pos(self):
        """Tagger POS en español entrenado con el corpus cess_esp (backoff)."""
        sents = [
            [(w, self._simplificar_tag(t)) for (w, t) in s if t]
            for s in cess_esp.tagged_sents()
        ]
        sents = [s for s in sents if s]
        base = DefaultTagger("NOUN")
        uni = UnigramTagger(sents, backoff=base)
        bi = BigramTagger(sents, backoff=uni)
        return TrigramTagger(sents, backoff=bi)

    @staticmethod
    def _features_ner(tokens, index, history):
        palabra = tokens[index]
        prev = tokens[index - 1] if index > 0 else "<INI>"
        sig = tokens[index + 1] if index < len(tokens) - 1 else "<FIN>"
        return {
            "palabra": palabra.lower(),
            "es_mayus_inicial": palabra[:1].isupper(),
            "es_todo_mayus": palabra.isupper(),
            "es_digito": palabra.isdigit(),
            "prefijo3": palabra[:3].lower(),
            "sufijo3": palabra[-3:].lower(),
            "prev": prev.lower(),
            "prev_mayus": prev[:1].isupper(),
            "sig": sig.lower(),
            "sig_mayus": sig[:1].isupper(),
            "inicio": index == 0,
            "prev_tag": history[index - 1] if index > 0 else "<INI>",
        }

    def _entrenar_ner(self):
        """Tagger NER (IOB) en español entrenado con conll2002 (esp)."""
        sents = [
            [(w, iob) for (w, _pos, iob) in s]
            for s in conll2002.iob_sents("esp.train")
        ]
        if self.max_sents_ner:
            sents = sents[: self.max_sents_ner]
        return ClassifierBasedTagger(
            train=sents, feature_detector=self._features_ner
        )

    # ------------------------------------------------------------------ pasos
    def cargar_datos(self):
        self.df = pd.read_csv(self.ruta_csv)
        if self.columna not in self.df.columns:
            raise KeyError(f"No existe la columna '{self.columna}' en el CSV")
        self.df[self.columna] = self.df[self.columna].fillna("").astype(str)
        return self.df

    def tokenizar(self, texto):
        return word_tokenize(texto, language="spanish")

    def etiquetar_pos(self, tokens):
        return self.tagger_pos.tag(tokens)

    def quitar_stopwords(self, tokens_pos):
        """Elimina stopwords y signos de puntuación conservando el tag POS."""
        return [
            (tok, pos) for tok, pos in tokens_pos
            if tok.lower() not in self.stop_es
            and pos != "PUNCT"
            and any(c.isalnum() for c in tok)
        ]

    def reconocer_entidades(self, tokens):
        """Devuelve lista de (entidad, tipo) agrupando etiquetas IOB."""
        if not tokens:
            return []
        etiquetas = self.tagger_ner.tag(tokens)
        entidades, actual, tipo = [], [], None
        for tok, iob in etiquetas:
            if iob and iob.startswith("B-"):
                if actual:
                    entidades.append((" ".join(actual), tipo))
                actual, tipo = [tok], iob[2:]
            elif iob and iob.startswith("I-") and actual:
                actual.append(tok)
            else:
                if actual:
                    entidades.append((" ".join(actual), tipo))
                actual, tipo = [], None
        if actual:
            entidades.append((" ".join(actual), tipo))
        return entidades

    @staticmethod
    def normalizar_caso(tokens):
        return [t.lower() for t in tokens]

    def lematizar(self, tokens):
        if _HAY_SIMPLEMMA:
            return [simplemma.lemmatize(t, lang="es") for t in tokens]
        return [self.stemmer.stem(t) for t in tokens]  # respaldo: stem

    # --------------------------------------------------------------- pipeline
    def procesar_texto(self, texto):
        tokens = self.tokenizar(texto)                          # 1. tokenización
        tokens_pos = self.etiquetar_pos(tokens)                 # 2. POS
        sin_stop = self.quitar_stopwords(tokens_pos)            # 3. stopwords
        tokens_filtrados = [t for t, _ in sin_stop]
        entidades = self.reconocer_entidades(tokens_filtrados)  # 4. NER
        minusculas = self.normalizar_caso(tokens_filtrados)     # 5. minúsculas
        lemas = self.lematizar(minusculas)                      # 6. lematización
        return {
            "tokens": tokens,
            "pos_tags": tokens_pos,
            "tokens_sin_stopwords": tokens_filtrados,
            "pos_sin_stopwords": sin_stop,
            "entidades": entidades,
            "tokens_minusculas": minusculas,
            "lemas": lemas,
        }

    def ejecutar(self, ruta_salida="resenas_es_procesado.csv"):
        if self.df is None:
            self.cargar_datos()
        resultados = self.df[self.columna].apply(self.procesar_texto)
        procesado = pd.DataFrame(resultados.tolist(), index=self.df.index)
        self.df = pd.concat([self.df, procesado], axis=1)
        self.df["texto_lematizado"] = self.df["lemas"].apply(" ".join)
        self.df.to_csv(ruta_salida, index=False, encoding="utf-8-sig")
        return self.df

## 3. Configuración e inicialización
La primera vez se entrenan los taggers POS y NER (puede tardar unos minutos); luego se cargan desde `cache_nltk/`.
Para acelerar el entrenamiento del NER puedes usar `max_sents_ner=2000`, por ejemplo.

In [ ]:
RUTA_CSV = "../processed/resenas_es.csv"
COLUMNA = "review_summary_es"
RUTA_SALIDA = "resenas_es_procesado.csv"

pipeline = NLTK_POS_Reviews(RUTA_CSV, columna=COLUMNA)

## 4. Demostración paso a paso sobre un texto de ejemplo

In [ ]:
texto = "Me encantó el hotel de Madrid, el personal fue muy amable y las habitaciones estaban limpias."

tokens = pipeline.tokenizar(texto)
print("1. Tokens:\n", tokens, "\n")

tokens_pos = pipeline.etiquetar_pos(tokens)
print("2. POS:\n", tokens_pos, "\n")

sin_stop = pipeline.quitar_stopwords(tokens_pos)
tokens_filtrados = [t for t, _ in sin_stop]
print("3. Sin stopwords:\n", sin_stop, "\n")

print("4. Entidades (NER):\n", pipeline.reconocer_entidades(tokens_filtrados), "\n")

minusculas = pipeline.normalizar_caso(tokens_filtrados)
print("5. Minúsculas:\n", minusculas, "\n")

print("6. Lemas:\n", pipeline.lematizar(minusculas))

## 5. Ejecución sobre el CSV completo

In [ ]:
df = pipeline.ejecutar(RUTA_SALIDA)
df[[COLUMNA, "pos_tags", "entidades", "lemas"]].head()

## 6. Exploración rápida de resultados

In [ ]:
from collections import Counter

# Palabras (lemas) más frecuentes
conteo = Counter(l for lemas in df["lemas"] for l in lemas)
pd.DataFrame(conteo.most_common(20), columns=["lema", "frecuencia"])

In [ ]:
# Entidades más frecuentes
ents = Counter(e for lista in df["entidades"] for e in lista)
pd.DataFrame(
    [(e[0], e[1], n) for e, n in ents.most_common(20)],
    columns=["entidad", "tipo", "frecuencia"],
)